# Project 3: **Ask‑the‑Web Agent**

Welcome to Project 3! In this project, you will learn how to use tool‑calling LLMs, extend them with custom tools, and build a simplified *Perplexity‑style* agent that answers questions by searching the web.

## Learning Objectives  
* Understand why tool calling is useful and how LLMs can invoke external tools.
* Implement a minimal loop that parses the LLM's output and executes a Python function.
* See how *function schemas* (docstrings and type hints) let us scale to many tools.
* Use **LangChain** to get function‑calling capability for free (ReAct reasoning, memory, multi‑step planning).
* Combine LLM with a web‑search tool to build a simple ask‑the‑web agent.

## Roadmap
1. Environment setup
2. Write simple tools and connect them to an LLM
3. Standardize tool calling by writing `to_schema`
4. Use LangChain to augment an LLM with your tools
5. Build a Perplexity‑style web‑search agent
6. (Optional) A minimal backend and frontend UI

# 1- Environment setup

## 1.1- Conda environment

Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook and run:

```bash
# Create and activate the conda environment
conda env create -f environment.yml && conda activate web_agent

# Register this environment as a Jupyter kernel
python -m ipykernel install --user --name=web_agent --display-name "web_agent"
```
Once this is done, you can select “web_agent” from the Kernel → Change Kernel menu in Jupyter or VS Code.


> Behind the scenes:
> * Conda reads `environment.yml`, resolves the pinned dependencies, creates an isolated environment named `web_agent`, and activates it.
> * `ollama pull` downloads the model so you can run it locally without API calls.


## 1.2 Ollama setup

In this project, we start with `gemma3-1B` because it is lightweight and runs on most machines. You can try other smaller or larger LLMs such as `mistral:7b`, `phi3:mini`, or `llama3.2:1b` to compare performance. Explore available models here: https://ollama.com/library

```bash
ollama pull gemma3:1b
```

`ollama pull` downloads the model so you can run it locally without API calls.


## 2- Tool Calling

LLMs are strong at answering questions, but they cannot directly access external data such as live web results, APIs, or computations. In real applications, agents rarely rely only on their internal knowledge. They need to query APIs, retrieve data, or perform calculations to stay accurate and useful. Tool calling bridges this gap by allowing the LLM to request actions from the outside world.


We describe each tool’s interface in the model’s prompt, defining what it does and what arguments it expects. When the model decides that a tool is needed, it emits a structured output like: `TOOL_CALL: {"name": "get_current_weather", "args": {"city": "San Francisco"}}`. Your code will detect this output, execute the corresponding function, and feed the result back to the LLM so the conversation continues.

In this section, you will implement a simple `get_current_weather` function and teach the `gemma3` model how to use it when required in four steps:
1. Implement the tool
2. Create the instructions for the LLM
3. Call the LLM with the prompt
4. Parse the LLM output and call the tool

In [61]:
# ---------------------------------------------------------
# Step 1: Implement the tool
# ---------------------------------------------------------
# Your goal: give the model a way to access weather information.
# You can either:
#   (a) Call a real weather API (for example, OpenWeatherMap), or
#   (b) Create a dummy function that returns a fixed response (e.g., "It is 23°C and sunny in San Francisco.")
#
# Requirements:
#   • The function should be named `get_current_weather`
#   • It should take two arguments:
#         - city: str
#         - unit: str = "celsius"
#   • Return a short, human-readable sentence describing the weather.
#
# Example expected behavior:
#   get_current_weather("San Francisco") → "It is 23°C and sunny in San Francisco."
#

import requests

def return_current_weather_in_words(city: str, unit: str):
    temp = int(unit)   # convert the input to number
    weather_description = ""

    match temp:
        case t if t >= 32:
            weather_description = f"It is {t}°C and extremely hot in {city}."
        case t if 27 <= t < 32:
            weather_description = f"It is {t}°C and hot and sunny in {city}."
        case t if 21 <= t < 27:
            weather_description = f"It is {t}°C and warm and pleasant in {city}."
        case t if 15 <= t < 21:
            weather_description = f"It is {t}°C and mildly cool in {city}."
        case t if 10 <= t < 15:
            weather_description = f"It is {t}°C and cool and breezy in {city}."
        case t if 4 <= t < 10:
            weather_description = f"It is {t}°C and cold in {city}."
        case t if -5 <= t < 4:
            weather_description = f"It is {t}°C and very cold in {city}."
        case t if t < -5:
            weather_description = f"It is {t}°C and freezing in {city}."
        case _:
            weather_description = f"Unable to classify the weather for {t}°C in {city}."

    return weather_description

def get_current_weather(city:str):
    """Return the current weather as human-readable sentence."""
    url = "http://api.weatherstack.com/current" 

    params = {
        "access_key": "674aa1057c94ebf35e98643f40fa3299",
        "query": city,
        "units": "m"   # m = metric (Celsius)
    }

    response = requests.get(url, params=params)

    unit = response.json()["current"]["temperature"]

    weather_description = return_current_weather_in_words(city, unit)

    # print(weather_description)

    return weather_description

    # return weather_description
    # return response.json()


def add(a:int, b:int):
    return a + b

    
# get_current_weather("Chicago")

# y = add(3, 5)

# print(y)


In [62]:
# ---------------------------------------------------------
# Step 2: Create the prompt for the LLM to call tools
# ---------------------------------------------------------
# Goal:
#   Build the system and user prompts that instruct the model when and how
#   to use your tool (`get_current_weather`).
#
# What to include:
#   • A SYSTEM_PROMPT that tells the model about the tool use and describe the tool
#   • A USER_QUESTION with a user query that should trigger the tool.
#       Example: "What is the weather in San Diego today?"

# Try experimenting with different system and user prompts
# ---------------------------------------------------------

# SYSTEM_PROMPT = """
# You are a helpful assistant. 
# When the user asks about weather, you MUST respond ONLY in the following format:

# TOOL_CALL:{"tool": "get_current_weather", "args": {"city": "<city>", "unit": "<unit>"}}

# Rules:
# - No natural language.
# - No code blocks.
# - No markdown.
# - No extra text before or after.
# - Only output TOOL_CALL JSON exactly.
# """


# SYSTEM_PROMPT = """
# You are a helpful assistant.

# RULES:
# 1. If the user asks about weather for one city → output:
#    TOOL_CALL:[{"tool":"get_current_weather","args":{"city":"<city>"}}]

# 2. If the user asks about weather for two cities → output:
#    TOOL_CALL:[
#      {"tool":"get_current_weather","args":{"city":"<city1>"}},
#      {"tool":"get_current_weather","args":{"city":"<city2>"}}
#    ]

# 3. And if the user asks for sum of 2 numbers, you MUST respond ONLY with:
#     TOOL_CALL:{"tool": "add", "args": {"a": "<a>", "b": "<b>"}}

# 4. After the tool result is returned, you may answer follow-up questions normally.

# 5. Do NOT call the tool for questions like:
#    - "Do I need an umbrella?"
#    - "Should I wear a jacket?"
#    - "Is it hot today?"

# Those questions should be answered normally using the weather info already provided by the tool result.

# """

SYSTEM_PROMPT = """You are a helpful assistant with access to tools.

RULES:
1. When user asks about weather, respond ONLY with:
   TOOL_CALL:{"tool":"get_current_weather","args":{"city":"<city>"}}

2. When user asks to add numbers, respond ONLY with:
   TOOL_CALL:{"tool":"add","args":{"a":<number>,"b":<number>}}

3. Output ONLY the TOOL_CALL line - no other text before or after.

4. After receiving TOOL_RESULT, answer the user's question naturally.

Examples:
User: "What is 5 + 3?"
Assistant: TOOL_CALL:{"tool":"add","args":{"a":5,"b":3}}

User: "What's the weather in Boston?"
Assistant: TOOL_CALL:{"tool":"get_current_weather","args":{"city":"Boston"}}
"""

USER_QUESTION = (
    # "What is the weather in San Diego today?"
    "what is 5 + 6?"
)


Now that you have defined a tool and shown the model how to use it, the next step is to call the LLM using your prompt.

Start the **Ollama** server in a terminal with `ollama serve`. This launches a local API endpoint that listens for LLM requests. Once the server is running, return to the notebook and in the next cell send a query to the model.


In [63]:
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")

# ---------------------------------------------------------
# Step 3: Call the LLM with your prompt
# ---------------------------------------------------------
# Task:
#   Send SYSTEM_PROMPT + USER_QUESTION to the model.
#
# Steps:
#   1. Use the Ollama client to create a chat completion. 
#       - You may find some examples here: https://platform.openai.com/docs/api-reference/chat/create
#       - If you are unsure, search the web for "client.chat.completions.create"
#   2. Print the raw response.
#
# Expected:
#   The model should return something like:
#   TOOL_CALL: {"name": "get_current_weather", "args": {"city": "San Diego"}}
# ---------------------------------------------------------


response = client.chat.completions.create(
    model="gemma3:1b",  
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_QUESTION}
    ]
)

model_output = response.choices[0].message.content

print(response)


ChatCompletion(id='chatcmpl-127', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='TOOL_CALL:{"tool":"add","args":{"a":5,"b":6}}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1764652525, model='gemma3:1b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=20, prompt_tokens=217, total_tokens=237, completion_tokens_details=None, prompt_tokens_details=None))


In [64]:
# ---------------------------------------------------------
# Step 4: Parse the LLM output and call the tool
# ---------------------------------------------------------
# Task:
#   Detect when the model requests a tool, extract its name and arguments,
#   and execute the corresponding function.
#
# Steps:
#   1. Search for the text pattern "TOOL_CALL:{...}" in the model output.
#   2. Parse the JSON inside it to get the tool name and args.
#   3. Call the matching function (e.g., get_current_weather).
#
# Expected:
#   You should see a line like:
#       Calling tool `get_current_weather` with args {'city': 'San Diego'}
#       Result: It is 23°C and sunny in San Diego.
# ---------------------------------------------------------

import re, json

#non-regex way

# marker = "TOOL_CALL:"

# if marker in model_output:
#     start = model_output.index(marker) + len(marker)
#     json_str = model_output[start:].strip()

#     # Parse the JSON object
#     tool_data = json.loads(json_str)
#     tool_name = tool_data.get("tool")
#     tool_args = tool_data.get("args", {})

#     print(f"Calling tool `{tool_name}` with args {tool_args}")

#     if tool_name == "get_current_weather":
#         result = get_current_weather(**tool_args)
#         print("Result:", result)

#regex way
pattern = r"TOOL_CALL:\s*(\{.*\})"   # capture the JSON object after TOOL_CALL:

match = re.search(pattern, model_output, re.DOTALL)

if match:
    json_str = match.group(1)

    tool_data = json.loads(json_str)
    tool_name = tool_data.get("tool")
    tool_args = tool_data.get("args", {})

    print(f"Calling tool `{tool_name}` with args {tool_args}")

    if tool_name == "get_current_weather":
        result = get_current_weather(**tool_args)
        print("Result:", result)

    elif tool_name == "add":
        a = int(tool_args.get("a"))
        b = int(tool_args.get("b"))
        result = add(a=a, b=b)
        print("Result:", result)
else:
    print("No tool call detected.")




Calling tool `add` with args {'a': 5, 'b': 6}
Result: 11


# 3- Standadize tool calling

So far, we handled tool calling manually by writing one regex and one hard-coded function. This approach does not scale if we want to add more tools. Adding more tools would mean more `if/else` blocks and manual edits to the `TOOL_SPEC` prompt.

To make the system flexible, we can standardize tool definitions by automatically reading each function’s signature, converting it to a JSON schema, and passing that schema to the LLM. This way, the LLM can dynamically understand which tools exist and how to call them without requiring manual updates to prompts or conditional logic.

Next, you will implement a small helper that extracts metadata from functions and builds a schema for each tool.

In [65]:
# ---------------------------------------------------------
# Generate a JSON schema for a tool automatically
# ---------------------------------------------------------
#
# Steps:
#   1. Use `inspect.signature` to get function parameters.
#   2. For each argument, record its name, type, and description.
#   3. Build a schema containing:
#   4. Test your helper on `get_current_weather` and print the result.
#
# Expected:
#   A dictionary describing the tool (its name, args, and types).
# ---------------------------------------------------------

from pprint import pprint
import inspect

def to_schema(fn):
    signature = inspect.signature(fn)
    parameters = signature.parameters

    parameters_schema = []

    for name, param_obj in parameters.items():
        param_info = {
            "name": name,
            "type": str(param_obj.annotation) if param_obj.annotation != inspect._empty else "unknown",
            "default": param_obj.default if param_obj.default != inspect._empty else None,
        }
        parameters_schema.append(param_info)

    return {
        "tool_name": fn.__name__,
        "description": fn.__doc__,
        "parameters": parameters_schema
    }

tool_schema = to_schema(get_current_weather)
pprint(tool_schema)

{'description': 'Return the current weather as human-readable sentence.',
 'parameters': [{'default': None, 'name': 'city', 'type': "<class 'str'>"}],
 'tool_name': 'get_current_weather'}


In [70]:
# ---------------------------------------------------------
# Provide the tool schema to the model
# ---------------------------------------------------------
# Goal:
#   Give the model a "menu" of available tools so it can choose
#   which one to call based on the user’s question.
#
# Steps:
#   1. Add an extra system message (e.g., name="tool_spec")
#      containing the JSON schema(s) of your tools.
#   2. Include SYSTEM_PROMPT and the user question as before.
#   3. Send the messages to the model (e.g., gemma3:1b).
#   4. Print the raw model output to see if it picks the right tool.
#
# Expected:
#   The model should produce a structured TOOL_CALL indicating
#   which tool to use and with what arguments.
# ---------------------------------------------------------

# ---------------------------------------------------------
# Provide the tool schema to the model
# ---------------------------------------------------------
import json


tool_schema = {
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Return current weather for a city in human-readable form.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string"}
                    },
                    "required": ["city"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "add",
                "description": "Add two integers.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "integer"},
                        "b": {"type": "integer"}
                    },
                    "required": ["a", "b"]
                }
            }
        }
    ]
}


# Build the tool spec message
tool_spec_message = {
    "role": "system",
    "name": "tool_spec",
    "content": json.dumps(tool_schema)   # send schema as JSON string
}

messages = [
    tool_spec_message,
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "what is 2 + 2?"},  #USER_QUESTION
    # {"role": "user", "content": "Is Boston warmer today or Seattle?"},  #USER_QUESTION
]

# Call the model
response = client.chat.completions.create(
    # model="gemma3:1b",
    model="llama3.2:latest",
    messages=messages
)

print("\n=== RAW MODEL OUTPUT ===")
print(response.choices[0].message.content)


# ---------------------------------------------------------
# Execute tool call automatically (minimal required code)
# ---------------------------------------------------------

msg = response.choices[0].message

# ---------------------------------------------------------
# Handle Ollama-style TOOL_CALL text manually
# ---------------------------------------------------------


import json

raw = msg.content.strip()

python_functions = {
    "get_current_weather": get_current_weather,
    "add": add,
}

tool_results = []

# Split by semicolon OR newline to handle multiple tool calls
tool_call_lines = []
for line in raw.split(';'):  # Split by semicolon first
    for subline in line.splitlines():  # Then by newline
        tool_call_lines.append(subline.strip())

# Process each tool call
for line in tool_call_lines:
    if line.startswith("TOOL_CALL:"):
        # Remove prefix and strip
        json_str = line[len("TOOL_CALL:"):].strip().rstrip(',')
        
        # Fix common model mistakes
        json_str = json_str.replace('"args=', '"args":')  # Fix missing colon
        
        try:
            tool_call = json.loads(json_str)
        except json.JSONDecodeError:
            print("Failed to parse JSON:", json_str)
            continue

        tool = tool_call["tool"]
        
        # Handle both formats - with "args" key or without
        if "args" in tool_call:
            args = tool_call["args"]
        else:
            # Extract all keys except "tool" as args
            args = {k: v for k, v in tool_call.items() if k != "tool"}

        print(f"\nCalling tool `{tool}` with args {args}")

        if tool in python_functions:
            try:
                result = python_functions[tool](**args)
                tool_results.append((tool, result))
                print(f"Result: {result}")
            except Exception as e:
                print(f"Error calling tool: {e}")
        else:
            print(f"No matching Python function for tool `{tool}`")

# Send all results back to the model
if tool_results:
    followup_messages = messages + [
        {
            "role": "assistant",
            "content": raw
        },
        {
            "role": "user",
            "content": f"TOOL_RESULTS: {json.dumps(tool_results)}"
        }
    ]

    followup = client.chat.completions.create(
        model="llama3.2:latest",
        messages=followup_messages
    )

    print("\n=== FINAL MODEL ANSWER ===")
    print(followup.choices[0].message.content)
else:
    print("\nNo tool call detected in the original model output.")


=== RAW MODEL OUTPUT ===
TOOL_CALL:{"tool":"add","args":{"a":2,"b":2}}

Calling tool `add` with args {'a': 2, 'b': 2}
Result: 4

=== FINAL MODEL ANSWER ===
The result of the calculation is 4. Would you like to try another math problem or ask about something else?


## 4- LangChain for Tool Calling
So far, you built a simple tool-calling pipeline manually. While this helps you understand the logic, it does not scale well when working with multiple tools, complex parsing, or multi-step reasoning.

LangChain simplifies this process. You only need to declare your tools, and its *Agent* abstraction handles when to call a tool, how to use it, and how to continue reasoning afterward.

In this section, you will use the **ReAct** Agent (Reasoning + Acting). It alternates between reasoning steps and tool use, producing clearer and more reliable results. We will explore reasoning-focused models in more depth next week.

The following links might be helpful:
- https://python.langchain.com/api_reference/langchain/agents/langchain.agents.initialize.initialize_agent.html
- https://python.langchain.com/docs/integrations/tools/
- https://python.langchain.com/docs/integrations/chat/ollama/
- https://python.langchain.com/api_reference/core/language_models/langchain_core.language_models.llms.LLM.html

In [71]:
# ---------------------------------------------------------
# Step 1: Define tools for LangChain
# ---------------------------------------------------------
# Goal:
#   Convert your weather function into a LangChain-compatible tool.
#
# Steps:
#   1. Import `tool` from `langchain.tools`.
#   2. Keep your existing `get_current_weather` helper as before.
#   3. Create a new function (e.g., get_weather) that calls it.
#   4. Add the `@tool` decorator so LangChain can register it automatically.
#
# Notes:
#   • The decorator converts your Python function into a standardized tool object.
#   • Start with keeping the logic simple and offline-friendly.

from langchain.tools import tool
import requests

# ---------------------------------------------------------
# Step 1: Define tools for LangChain
# ---------------------------------------------------------

# Your original helper function
def return_current_weather_in_words(city: str, unit: str):
    temp = int(unit)   # convert the input to number
    weather_description = ""
    match temp:
        case t if t >= 32:
            weather_description = f"It is {t}°C and extremely hot in {city}."
        case t if 27 <= t < 32:
            weather_description = f"It is {t}°C and hot and sunny in {city}."
        case t if 21 <= t < 27:
            weather_description = f"It is {t}°C and warm and pleasant in {city}."
        case t if 15 <= t < 21:
            weather_description = f"It is {t}°C and mildly cool in {city}."
        case t if 10 <= t < 15:
            weather_description = f"It is {t}°C and cool and breezy in {city}."
        case t if 4 <= t < 10:
            weather_description = f"It is {t}°C and cold in {city}."
        case t if -5 <= t < 4:
            weather_description = f"It is {t}°C and very cold in {city}."
        case t if t < -5:
            weather_description = f"It is {t}°C and freezing in {city}."
        case _:
            weather_description = f"Unable to classify the weather for {t}°C in {city}."
    return weather_description

# LangChain-compatible tool using @tool decorator
@tool
def get_weather(city: str) -> str:
    """Return the current weather as human-readable sentence for a given city."""
    url = "http://api.weatherstack.com/current" 
    params = {
        "access_key": "674aa1057c94ebf35e98643f40fa3299",
        "query": city,
        "units": "m"   # m = metric (Celsius)
    }
    response = requests.get(url, params=params)
    unit = response.json()["current"]["temperature"]
    weather_description = return_current_weather_in_words(city, str(unit))
    return weather_description

@tool
def add(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

# Test the tools
if __name__ == "__main__":
    # Test weather tool
    print(get_weather.invoke({"city": "Boston"}))
    
    # Test add tool
    print(add.invoke({"a": 5, "b": 3}))
    
    # Show tool information
    print("\nTool name:", get_weather.name)
    print("Tool description:", get_weather.description)
    print("Tool args:", get_weather.args)

It is -1°C and very cold in Boston.
8

Tool name: get_weather
Tool description: Return the current weather as human-readable sentence for a given city.
Tool args: {'city': {'title': 'City', 'type': 'string'}}


In [73]:
# ---------------------------------------------------------
# Step 2: Initialize the LangChain Agent
# ---------------------------------------------------------
# Goal:
#   Connect your tool to a local LLM using LangChain’s ReAct-style agent.
#
# Steps:
#   1. Import the required classes:
#        - ChatOllama (for local model access)
#        - initialize_agent, Tool, AgentType
#   2. Create an LLM instance (e.g., model="gemma3:1b", temperature=0).
#   3. Add your tool(s) to a list
#   4. Initialize the agent using initialize_agent
#   5. Test the agent with a natural question (e.g., "Do I need an umbrella in Seattle today?").
#
# Expected:
#   The model should reason through the question, call your tool,
#   and produce a final answer in plain language.
# ---------------------------------------------------------

from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, Tool, AgentType

# ---------------------------------------------------------
# Step 2: Initialize the LangChain Agent
# ---------------------------------------------------------
import requests
from langchain.tools import tool

# ---------------------------------------------------------
# YOUR CODE HERE (~5 lines of code)
# ---------------------------------------------------------

# 1. Create LLM instance
llm = ChatOllama(model="llama3.2:latest", temperature=0)

# 2. Create list of tools
tools = [get_weather, add]

# 3. Initialize agent with STRUCTURED_CHAT (supports multi-input tools)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

# 4. Test the agent
response = agent.invoke({"input": "Do I need an umbrella in Seattle today?"})
print("\n=== FINAL ANSWER ===")
print(response["output"])





> Entering new AgentExecutor chain...
Action:
```
{
  "action": "get_weather",
  "action_input": {"city": "Seattle"}
}
```


Observation: It is 7°C and cold in Seattle.
Thought:Action:
```
{
  "action": "get_weather",
  "action_input": {"city": "Seattle"}
}
```


Observation: It is 7°C and cold in Seattle.
Thought:Action:
```
{
  "action": "Final Answer",
  "action_input": "You should consider bringing an umbrella, as it's cold and rainy in Seattle today."
}
```

> Finished chain.

=== FINAL ANSWER ===
You should consider bringing an umbrella, as it's cold and rainy in Seattle today.


### What just happened?

The console log displays the **Thought → Action → Observation → …** loop until the agent produces its final answer. Because `verbose=True`, LangChain prints each intermediate reasoning step.

If you want to add more tools, simply append them to the tools list. LangChain will handle argument validation, schema generation, and tool-calling logic automatically.


## 5- Perplexity‑Style Web Search
Agents become much more powerful when they can look up real information on the web instead of relying only on their internal knowledge.

In this section, you will combine everything you have learned to build a simple Ask-the-Web Agent. You will integrate a web search tool (DuckDuckGo) and make it available to the agent using the same tool-calling approach as before.

This will let the model retrieve fresh results, reason over them, and generate an informed answer—similar to how Perplexity works.

You may find some examples from the following links:
- https://pypi.org/project/duckduckgo-search/

In [74]:
# ---------------------------------------------------------
# Step 1: Add a web search tool
# ---------------------------------------------------------
# Goal:
#   Create a tool that lets the agent search the web and return results.
#
# Steps:
#   1. Use DuckDuckGo for quick, open web searches.
#   2. Write a helper function (e.g., search_web) that:
#        • Takes a query string
#        • Uses DDGS to fetch top results (titles + URLs)
#        • Returns them as a formatted string
#   3. Wrap it with the @tool decorator to make it available to LangChain.


from ddgs import DDGS
from langchain.tools import tool

# ---------------------------------------------------------
# Step 1: Add a web search tool
# ---------------------------------------------------------

@tool
def search_web(query: str) -> str:
    """Search the web using DuckDuckGo and return top results with titles and URLs."""
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)
        
        if not results:
            return "No results found."
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            title = result.get('title', 'No title')
            url = result.get('href', 'No URL')
            snippet = result.get('body', 'No description')
            formatted_results.append(f"{i}. {title}\n   URL: {url}\n   {snippet}\n")
        
        return "\n".join(formatted_results)
    except Exception as e:
        return f"Error searching web: {str(e)}"

# Test the tool
if __name__ == "__main__":
    result = search_web.invoke({"query": "latest AI developments 2024"})
    print(result)

1. Unlock AI’s Value Securely - AI Innovation
   URL: https://www.bing.com/aclick?ld=e8iNv2BWlmh_Lx5UWVOmguLjVUCUy0fyqK99_35uNbJt6YKs95MWyf5LFtEkLN_OfcXXTHW3X3Zi8YBD2WRSweCCx4t0cBpyBAlI_DkjRCOBCjwKL-RWlE9ZSyWGU9vVbZ2A0ddpe1k6BZk65j6ny_m3W7vMSpUJM643FJNwTO9yJCIxQjNvElQPfK4KbN-sTaSxjJVm5TNluk6BJJxOyAI4qRXlc&u=aHR0cHMlM2ElMmYlMmZ3d3cudGVjaHRhcmdldC5jb20lMmZzZWFyY2hhd3MlMmZQb3dlcm9mQUl3aXRoQVdTJTJmSWRlbnRpdHktQ2FuLUhlbHAtWW91LVVubG9jay10aGUtVmFsdWUtb2YtQUklM2Z1dG1fc291cmNlJTNkYmluZyUyNmludCUzZG9mZiUyNnByZSUzZG9mZiUyNnV0bV9tZWRpdW0lM2RjcGMlMjZ1dG1fdGVybSUzZEdBVyUyNnV0bV9jb250ZW50JTNkc3lfbHAxMDMxMjAyNUdPT0dPVEhSX0dzaWRzQVdTX0FXU19FbWJlZF9JTzMzNTMyNiUyNnV0bV9jYW1wYWlnbiUzZEFXU19FbWJlZF9Qb3dlck9mQUlfTkElMjZPZmZlciUzZHN5X2xwMTAzMTIwMjVHT09HT1RIUl9Hc2lkc0FXU19BV1NfRW1iZWRfSU8zMzUzMjYlMjZtc2Nsa2lkJTNkNTc5NTA4MzhiYjk4MTY3NmM5MmM2MTE2YzEwMjE5NmE&rlid=57950838bb981676c92c6116c102196a
   techtarget.com has been visited by 100K+ users in the past month Explore identity patterns to drive safe and effec

In [75]:

# ---------------------------------------------------------
# Step 2: Initialize the web-search agent
# ---------------------------------------------------------
# Goal:
#   Connect your `web_search` tool to a language model
#   so the agent can search and reason over real data.
#
# Steps:
#   1. Import `initialize_agent` and `AgentType`.
#   2. Create an LLM (e.g., ChatOllama).
#   3. Add your `web_search` tool to the tools list.
#   4. Initialize the agent using: initialize_agent
#   5. Keep `verbose=True` to observe reasoning steps.
#
# Expected:
#   The agent should be ready to accept user queries
#   and use your web search tool when needed.
# ---------------------------------------------------------
from langchain.agents import initialize_agent, AgentType
from langchain.llms import OpenAI
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="llama3.2:latest", temperature=0)
tools = [search_web]
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Test it
agent.invoke({"input": "What are the latest AI developments?"})




> Entering new AgentExecutor chain...
Thought: To find the latest AI developments, I should search the web using DuckDuckGo.

Action: search_web
Action Input: "latest AI developments"
Observation: 1. The Latest AI News and AI Breakthroughs that Matter Most: 2025 | News
   URL: 
   

2. Google News - Artificial intelligence - Latest
   URL: https://news.google.com/topics/CAAqJAgKIh5DQkFTRUFvSEwyMHZNRzFyZWhJRlpXNHRSMElvQUFQAQ
   Valve artist says that's "like saying food products shouldn't have their ingredients list" as Epic's Tim Sweeney and more call on Steam to drop the 'Made with AI' label

3. The Top Artificial Intelligence Trends | IBM
   URL: https://www.ibm.com/think/insights/artificial-intelligence-trends
   Given the breadth and depth of AIdevelopment, no roundup of AItrends can hope to be exhaustive. This piece is no exception. We’ve narrowed things down to a list of 10: 5 developments that have driven the first half of the year, and 5 more that we expect to play a major ro

{'input': 'What are the latest AI developments?',
 'output': 'The latest AI developments include recent breakthroughs in artificial intelligence news, trends, and technologies, such as advancements in machine learning, natural language processing, and computer vision. These developments have significant implications for industries like healthcare, finance, and transportation, and are expected to shape the future of work and daily life.'}


Let’s see the agent's output in action with a real example.


In [77]:

# ---------------------------------------------------------
# Step 3: Test your Ask-the-Web agent
# ---------------------------------------------------------
# Goal:
#   Verify that the agent can search the web and return
#   a summarized answer based on real results.
#
# Steps:
#   1. Ask a natural question that requires live information,
#      for example: "What are the current events in San Francisco this week?"
#   2. Call agent.
#
# Expected:
#   The agent should call `web_search`, retrieve results,
#   and generate a short summary response.
# ---------------------------------------------------------

# ---------------------------------------------------------
# Step 3: Test your Ask-the-Web agent
# ---------------------------------------------------------

# # Test 1: Current events
# print("="*60)
# print("TEST 1: Current Events")
# print("="*60)
# question = "What are the current events in San Francisco this week?"
# response = agent.invoke({"input": question})
# print(response["output"])

# Test 2: Latest technology news
print("\n" + "="*60)
print("TEST 2: Technology News")
print("="*60)
question = "What are the latest breakthroughs in quantum computing?"
response = agent.invoke({"input": question})
print(response["output"])

# # Test 3: Current prices/data
# print("\n" + "="*60)
# print("TEST 3: Current Data")
# print("="*60)
# question = "What is the current price of Bitcoin?"
# response = agent.invoke({"input": question})
# print(response["output"])

# # Test 4: Recent sports results
# print("\n" + "="*60)
# print("TEST 4: Sports Results")
# print("="*60)
# question = "Who won the latest NBA championship?"
# response = agent.invoke({"input": question})
# print(response["output"])

# # Test 5: Product recommendations
# print("\n" + "="*60)
# print("TEST 5: Product Search")
# print("="*60)
# question = "What are the best laptops for programming in 2024?"
# response = agent.invoke({"input": question})
# print(response["output"])




TEST 2: Technology News


> Entering new AgentExecutor chain...
Thought: To find the latest breakthroughs in quantum computing, I should search the web using DuckDuckGo.

Action: search_web
Action Input: "latest breakthroughs in quantum computing"
Observation: 1. IBM Delivers New Quantum Processors, Software, and Algorithm ...
   URL: https://newsroom.ibm.com/2025-11-12-ibm-delivers-new-quantum-processors,-software,-and-algorithm-breakthroughs-on-path-to-advantage-and-fault-tolerance
   Nov 12, 2025 · IBM unveiled fundamental progress on its path to delivering both quantum advantage by the end of 2026 and fault-tolerant quantumcomputing by 2029.

2. Microsoft unveils Majorana 1, the world’s first quantum ...
   URL: https://azure.microsoft.com/en-us/blog/quantum/2025/02/19/microsoft-unveils-majorana-1-the-worlds-first-quantum-processor-powered-by-topological-qubits/
   Feb 19, 2025 · Microsoft announces a breakthrough in quantum computing with Majorana 1, the world’s first QPU powered


## 6- A minimal UI
This project includes a simple **React** front end that sends the user’s question to a FastAPI back end and streams the agent’s response in real time. To run the UI:

1- Open a terminal and start the Ollama server: `ollama serve`.

2- In a second terminal, navigate to the frontend folder and install dependencies:`npm install`.

3- In the same terminal, navigate to the backend folder and start the FastAPI back‑end: `uvicorn app:app --reload --port 8000`

4- Open a third terminal, navigate to the frontend folder, and start the React dev server: `npm run dev`

5- Visit `http://localhost:5173/` in your browser.



## 🎉 Congratulations!

* You have built a **web‑enabled agent**: tool calling → JSON schema → LangChain ReAct → web search → simple UI.
* Try adding more tools, such as news or finance APIs.
* Experiment with multiple tools, different models, and measure accuracy vs. hallucination.


👏 **Great job!** Take a moment to celebrate. The techniques you implemented here power many production agents and chatbots.